In [4]:
import os
os.chdir('/home/gianfranco/projects/2025/Visual_Intelligence_Project/Mixture-of-Recursions-for-Vision')  # repo root

from omegaconf import OmegaConf
from util.config import preprocess_config
from model.util import load_model_from_config
from model.sharing_strategy import SHARING_STRATEGY
from util.misc import print_trainable_parameters

# Pick one:
cfg = OmegaConf.load('/home/gianfranco/projects/2025/Visual_Intelligence_Project/Mixture-of-Recursions-for-Vision/conf/pretrain_vision/250720_pretrain_smollm-135m_rec3_middle_cycle_random_lr3e-3_mor_token_linear_alpha_0.1_sigmoid_aux_loss_0.001_shared_vocab.yaml')
# cfg = OmegaConf.load('conf/pretrain_vision/smoke_50steps_token_multimodal.yaml')

cfg = preprocess_config(cfg)
model = load_model_from_config(cfg)

if cfg.recursive.get("enable"):
    model, _ = SHARING_STRATEGY[cfg.model](cfg, model)
if "kv_sharing" in cfg and cfg.kv_sharing.get("enable"):
    model.set_kv_sharing_config(cfg)
if "mor" in cfg and cfg.mor.get("enable"):
    if cfg.mor.type == "expert":
        model.transform_layer_to_mor_expert(cfg)
    elif cfg.mor.type == "token":
        model.transform_layer_to_mor_token(cfg)

print_trainable_parameters(model)

-------------------------------Preprocess Config -------------------------------
Automatically determining batch size based on `total_batch_size`
total_batch_size              : 256 (given)
torch.cuda.device_count()     : 1
per_device_train_batch_size   : 32 (given)
gradient_accumulation_steps   : 8 (computed)
actual total batch size       : 256
Setting output_dir  : smoke_50steps_token
Using deepspeed config = /home/gianfranco/projects/2025/Visual_Intelligence_Project/Mixture-of-Recursions-for-Vision/ds_configs/stage2.config
--------------------------------------------------------------------------------
Initializing model from scratch...
Using custom config for vanilla model...
 num_hidden_layers: 29
 vocab_size: 242271
trainable params: 179,156,736 || all params: 179,156,736 || relaxation params: 0 || trainable%: 100.0000
